Bronze data processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/akhildataengineer@hotmail.com/Atlikon_migration_pipeline/1_setup/3_utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-store-dp1/{data_source}/*.csv' #s3://sportsbar-store-dp/customers/*.csv
base_path

In [0]:
dbutils.fs.ls('s3://sportsbar-store-dp1/')

In [0]:
df_raw = spark.read\
        .format('csv')\
        .option('header', True)\
        .option('inferSchema', True)\
        .load(base_path)\
        .withColumn("read_timestamp", F.current_timestamp())\
        .select("*", "_metadata.file_name", "_metadata.file_size")

display(df.limit(10))

In [0]:
df_raw.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

Silver Data Processing

In [0]:
df_raw.printSchema()

In [0]:
# Find Duplicates
df_raw.groupBy("customer_id").count().filter(F.col("count") > 1).show()

# Drop Duplicates
df_enr = df_raw.dropDuplicates(["customer_id"])


In [0]:
# Find if there are any leading spaces
display(df_enr.filter(F.col("customer_name") != F.trim(F.col("customer_name"))))

# Remove leading spaces 
df_enr = df_enr.withColumn("customer_name", F.trim(F.col("customer_name")))

In [0]:
# Check if city names are correct
df_enr.select('city').distinct().show()

# typos -> correct names mapping
city_mapping = {
    "Bengaluruu": "Bengaluru",
    "Bengalore" : "Bengaluru",
    
    "Hyderabadd": "Hyderabad",
    "Hyderbad"  : "Hyderabad",

    "NewDelhi"   : "New Delhi",
    "NewDheli"   : "New Delhi",
    "NewDelhee"  : "New Delhi"
}

# Create list of allowed cities
allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

# Replace typos with correct names and remove cities not in allowed list
df_enr = df_enr\
            .replace(city_mapping, subset = ["city"])\
            .withColumn("city", F.when(F.col("city").isNull(), None)\
                                 .when(F.col("city").isin(allowed), F.col("city"))\
                                 .otherwise(None)
            )

In [0]:
# Check if customer names are correctly populated
df_enr.select('customer_name').distinct().show()

# Title case fix using initcap function
df_enr = df_enr\
            .withColumn("customer_name", F.when(F.col("customer_name").isNull(), None)\
                                          .otherwise(F.initcap(F.col("customer_name")))
            )

In [0]:
# Handling NULLs: city
df_enr.filter(F.col("city").isNull()).show(truncate = False)

# Null customers list and find its data
null_customers = ["Zenathlete Foods", "Sprintx Nutrition", "Primefuel Nutrition", "Recovery Lane "]
df_enr.filter(F.col("customer_name").isin(null_customers)).show(truncate = False)